# PyoxigraphExplorer Guide

This notebook demonstrates how to use `PyoxigraphExplorer`, a pyoxigraph-backed RDF graph explorer for the AI Atlas Nexus knowledge graph.

## Overview

`PyoxigraphExplorer` provides:
- **Drop-in compatibility** with `AtlasExplorer` — returns typed Pydantic objects
- **Efficient graph queries** via an indexed RDF store
- **Raw SPARQL access** for advanced queries
- **Natural language queries** — translate plain English to SPARQL via an LLM
- **Hybrid architecture** — RDF store for traversal, Pydantic objects for data

## Setup

In [1]:
from ai_atlas_nexus import AIAtlasNexus
from ai_atlas_nexus.blocks.graph_explorer import PyoxigraphExplorer, AtlasExplorer

# Load the ontology
nexus = AIAtlasNexus()
print(f"Loaded ontology with {len(nexus.get_all_risks())} risks")

/Users/ingevejs/Documents/workspace/ingelise/risk-atlas-nexus/v-ai-atlas-nexus/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[2026-05-25 20:40:23:589] - INFO - AIAtlasNexus - Created AIAtlasNexus instance. Base_dir: None


Loaded ontology with 546 risks


## Part 1: Using OxigraphExplorer Standalone

You can create an `OxigraphExplorer` instance directly from a Container object:

In [ ]:
# Create PyoxigraphExplorer instance
ox = PyoxigraphExplorer(nexus._ontology)
print("PyoxigraphExplorer initialized")

# Let's create an AtlasExplorer instance so we can compare them
atlas = AtlasExplorer(nexus._ontology)
print("AtlasExplorer initialized")

OxigraphExplorer initialized
AtlasExplorer initialized


### Basic Queries

In [3]:
# Get all classes in the knowledge graph
classes = ox.get_all_classes()
print(f"Available classes ({len(classes)}): {classes[:5]}...")
#print(classes)

Available classes (34): ['Action', 'Adapter', 'AiEval', 'AiEvalResult', 'AiTask']...


In [25]:
# Get all risks by collection key
risks = ox.get_all("risks")
print(f"Retrieved {len(risks)} risks via collection key 'risks'")

# Show first risk (returns typed Pydantic object)
if risks:
    first_risk = risks[0]
    print(f"\nFirst risk:")
    print(f"  Type: {type(first_risk).__name__}")
    print(f"  ID: {first_risk.id}")
    print(f"  Name: {first_risk.name}")
    print(f"  Taxonomy: {first_risk.isDefinedByTaxonomy}")

Retrieved 546 risks via collection key 'risks'

First risk:
  Type: Risk
  ID: atlas-evasion-attack
  Name: Evasion attack
  Taxonomy: ibm-risk-atlas


In [26]:
# You can also request by class name (singular or plural)
risks_by_class = ox.get_all("Risk")
risks_by_class_plural = ox.get_all("Risks")

print(f"Risks by class name 'Risk': {len(risks_by_class)}")
print(f"Risks by class name 'Risks': {len(risks_by_class_plural)}")
print(f"Match collection key results: {len(risks_by_class) == len(risks) and len(risks_by_class_plural) == len(risks)}")

Risks by class name 'Risk': 546
Risks by class name 'Risks': 546
Match collection key results: True


In [6]:
# Get all actions
actions = ox.get_all("actions")
actions_by_class = ox.get_all("Action")

print(f"Retrieved {len(actions)} actions via collection key")
print(f"Retrieved {len(actions_by_class)} actions via class name")
print(f"Match: {len(actions) == len(actions_by_class)}")

Retrieved 1085 actions via collection key
Retrieved 1085 actions via class name
Match: True


In [7]:
# Get by ID
risk_id = risks[0].id
retrieved = ox.get_by_id(None, risk_id)

print(f"Retrieved risk by ID: {retrieved.id}")
print(f"  Name: {retrieved.name}")
print(f"  Match: {retrieved.id == risk_id}")

Retrieved risk by ID: atlas-evasion-attack
  Name: Evasion attack
  Match: True


In [27]:
# Filter by attribute
nist_actions = ox.get_by_attribute("actions", "isDefinedByTaxonomy", "nist-ai-rmf")
print(f"Actions from NIST AI RMF taxonomy: {len(nist_actions)}")

if nist_actions:
    print(f"\nFirst NIST action:")
    print(f"  ID: {nist_actions[0].id}")
    print(f"  Name: {nist_actions[0].name}")

Actions from NIST AI RMF taxonomy: 212

First NIST action:
  ID: MG-4.3-003
  Name: MG-4.3-003


In [30]:
# Filter by multiple criteria 
filtered = ox.filter_instances(
    "risks",
    {
        "isDefinedByTaxonomy": "ibm-risk-atlas",
        "name":"Evasion attack"
    }
)
print(f"Filtered risks (IBM Risk Atlas): {len(filtered)}")

Filtered risks (IBM Risk Atlas): 1


### Advanced: Raw SPARQL Queries

The `sparql_query()` method gives you direct access to the RDF graph:

In [10]:
# Find all risks in the knowledge graph via SPARQL
sparql = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX nexus: <https://ibm.github.io/ai-atlas-nexus/ontology/>

SELECT ?s WHERE {
    ?s rdf:type nexus:Risk .
}
LIMIT 5
"""

results = ox.sparql_query(sparql)
print(f"First 5 risks (raw SPARQL results):")
for r in results:
    uri = r['?s']
    # Extract ID from URI
    risk_id = uri.split('/')[-1].rstrip('>')
    print(f"  {risk_id}")

First 5 risks (raw SPARQL results):
  credo-risk-049
  credo-risk-048
  credo-risk-047
  credo-risk-046
  credo-risk-045


In [11]:
# Find relationships: which actions target which risks?
sparql_relationships = """
PREFIX nexus: <https://ibm.github.io/ai-atlas-nexus/ontology/>

SELECT ?action ?risk WHERE {
    ?action nexus:hasRelatedRisk ?risk .
}
LIMIT 5
"""

relationships = ox.sparql_query(sparql_relationships)
print(f"Action-Risk relationships (sample):")
for rel in relationships:
    action_id = rel['?action'].split('/')[-1].rstrip('>')
    risk_id = rel['?risk'].split('/')[-1].rstrip('>')
    print(f"  {action_id} -> {risk_id}")

Action-Risk relationships (sample):
  ibm-factuality-adapter-granite-3.2-8b-instruct-alora-query-rewrite -> granite-relevance
  ibm-factuality-intrinsic-uncertainty -> granite-relevance
  ibm-factuality-intrinsic-requirement-check -> granite-relevance
  ibm-factuality-intrinsic-context-attribution -> granite-relevance
  ibm-factuality-intrinsic-policy-guardrails -> nist-governance


## Part 2: Comparing OxigraphExplorer with the Library

The `AIAtlasNexus` library uses `AtlasExplorer` internally for optimization. You can use `PyoxigraphExplorer` directly for the same data with graph query capabilities:

In [ ]:
# Compare library methods with direct explorer usage
ox = PyoxigraphExplorer(nexus._ontology)
atlas = AtlasExplorer(nexus._ontology)

print("Comparing risk retrieval methods:")
print()

# Library method
lib_risks = nexus.get_all_risks()
print(f"1. nexus.get_all_risks():")
print(f"   Returns {len(lib_risks)} risks")
print()

# Direct PyoxigraphExplorer
ox_risks = ox.get_all("risks")
print(f"2. PyoxigraphExplorer.get_all('risks'):")
print(f"   Returns {len(ox_risks)} risks")
print()

# Direct PyoxigraphExplorer by class name
ox_risks_class = ox.get_all("Risk")
print(f"3. PyoxigraphExplorer.get_all('Risk') [by class name]:")
print(f"   Returns {len(ox_risks_class)} risks")
print()

# Direct AtlasExplorer
atlas_risks = atlas.get_all("risks")
print(f"4. AtlasExplorer.get_all('risks'):")
print(f"   Returns {len(atlas_risks)} risks")
print()

# Verify results match for critical lookups
print("Verification - get_by_id consistency:")
test_id = lib_risks[0].id
ox_obj = ox.get_by_id(None, test_id)
atlas_obj = atlas.get_by_id(None, test_id)
lib_obj = nexus.get_risk(id=test_id)

print(f"  Risk ID: {test_id}")
print(f"  PyoxigraphExplorer found: {ox_obj is not None}")
print(f"  AtlasExplorer found: {atlas_obj is not None}")
print(f"  Library found: {lib_obj is not None}")
print(f"  All match: {ox_obj is not None and atlas_obj is not None and lib_obj is not None}")

Comparing risk retrieval methods:

1. nexus.get_all_risks():
   Returns 546 risks

2. OxigraphExplorer.get_all('risks'):
   Returns 546 risks

3. OxigraphExplorer.get_all('Risk') [by class name]:
   Returns 546 risks

4. AtlasExplorer.get_all('risks'):
   Returns 546 risks

Verification - get_by_id consistency:
  Risk ID: atlas-evasion-attack
  OxigraphExplorer found: True
  AtlasExplorer found: True
  Library found: True
  All match: True


## Part 3: Why Use PyoxigraphExplorer?
It could be suitable for adding more complex graph SPARQL queries

In [13]:
# Example: Complex graph query via SPARQL
# "Find all risks that are mitigated by actions in the NIST AI RMF"

sparql_complex = """
PREFIX nexus: <https://ibm.github.io/ai-atlas-nexus/ontology/>

SELECT ?risk ?action WHERE {
    ?action nexus:hasRelatedRisk ?risk .
    ?action nexus:isDefinedByTaxonomy "nist-ai-rmf" .
}
LIMIT 10
"""

print("Complex query: Risks mitigated by NIST AI RMF actions")
results = ox.sparql_query(sparql_complex)
print(f"Found {len(results)} risk-action pairs:")

for i, rel in enumerate(results[:3], 1):
    risk_id = rel['?risk'].split('/')[-1].rstrip('>')
    action_id = rel['?action'].split('/')[-1].rstrip('>')
    print(f"  {i}. Risk: {risk_id} <- Action: {action_id}")

Complex query: Risks mitigated by NIST AI RMF actions
Found 10 risk-action pairs:
  1. Risk: nist-information-security <- Action: MG-4.3-003
  2. Risk: nist-data-privacy <- Action: MG-4.3-003
  3. Risk: nist-information-integrity <- Action: MG-4.3-002


## Summary

`PyoxigraphExplorer` can be used alternative to `AtlasExplorer` that combines RDF graph capabilities with Pydantic object return types. This guide has covered:

**Core Functionality:**
- Basic queries: `get_all()`, `get_by_id()`, `get_by_attribute()`
- Multi-criteria filtering: `filter_instances()`, `query()` with kwargs
- Graph queries: `sparql_query()` for complex relationships
- Natural language queries: `natural_language_query()` for LLM-powered SPARQL generation
- Semantic similarity: `object_similarity()`, `object_equivalence()`

**Utility Methods:**
- `get_attribute()` — Retrieve single field from an object
- `filter_ids_by_type()` — Exclude IDs by type
- `arrange_ids_by_type()` — Organize IDs by entity type
- `clear_cache()` — Cache management

**Performance:**
- Query results are cached by parameter combination
- Use `get_by_id()` for fast O(1) lookups
- Use `get_by_attribute()` for single-attribute SPARQL filtering
- Use `sparql_query()` for complex relationship traversals
- Use `natural_language_query()` when you don't want to write SPARQL by hand
- Use `filter_instances()` or `query()` for multi-criteria AND filtering

**Use Cases:**

| Use Case | Method | Reason |
|----------|--------|--------|
| Get single object | `get_by_id()` | O(1) in-memory lookup |
| Get all of one type | `get_all()` | Cached, full table scan |
| Filter by one attribute | `get_by_attribute()` | SPARQL index query |
| Filter by multiple attributes | `filter_instances()` / `query()` | Python AND logic across fields |
| Find relationships | `sparql_query()` | Graph traversal, joins, patterns |
| Query in plain English | `natural_language_query()` | LLM translates to SPARQL |
| Compare objects | `object_similarity()` | Semantic similarity with embeddings |
| Organize IDs by type | `arrange_ids_by_type()` | Batch processing by type |

## Part 6: Performance Considerations

When to use different query approaches for optimal performance:

In [ ]:
print("""
QUERY STRATEGY GUIDE:
=====================

1. get_by_id(None, id) — O(1) lookup
   ✓ Use for: Single object lookups by ID
   ✓ Best for: Finding specific items when you know the ID
   Example: ox.get_by_id(None, "ibm-risk-atlas-001")

2. get_all(class_name) — Full table scan
   ✓ Use for: Retrieving all instances of a type
   ✓ Cached after first call
   ✓ Best for: Workloads that need many/all items from a class
   Example: ox.get_all("risks")

3. get_by_attribute(class_name, attr, value) — SPARQL index query
   ✓ Use for: Finding instances with specific attribute values
   ✓ Best for: Filtering on one attribute (single index)
   Example: ox.get_by_attribute("actions", "isDefinedByTaxonomy", "nist-ai-rmf")

4. filter_instances(class_name, filters) or query(class_name, **kwargs) — Python filter
   ✓ Use for: Multi-criteria AND filtering
   ✓ Best for: Complex conditions across multiple attributes
   ✓ Note: Fetches all instances, then filters in Python
   Example: ox.filter_instances("risks", {"isDefinedByTaxonomy": "ibm", "descriptor": "attack"})

5. sparql_query(query_str) — Custom graph traversal
   ✓ Use for: Complex relationships (joins, paths, aggregations)
   ✓ Best for: Finding connected entities, relationship patterns
   ✓ Most powerful but requires SPARQL knowledge
   Example: ox.sparql_query("SELECT ?risk WHERE {?action nexus:hasRelatedRisk ?risk}")

CACHING BEHAVIOR:
=================

• Query Results: Cached by (class_names, taxonomies, vocabulary, document)
  - Cache key includes ALL parameters
  - Different parameters = new cache entry
  - Identical parameters = instant cache hit

• ID Lookups: NOT cached (already O(1) in-memory dict)
  - get_by_id() always hits the id_cache directly
  - No cache invalidation needed for lookups

• When to call clear_cache():
  - If the underlying data changed (loaded new ontology)
  - To free memory if PyoxigraphExplorer is long-lived
  - Before/after benchmarking query performance
  - Note: ID cache persists (needed for proper object resolution)
""")

In [ ]:
print("""
=== object_similarity ===
Measure how similar two objects are by comparing all fields.

Signature:
  object_similarity(entry_a, entry_b, threshold, property_scores, max_depth=1, weight_dict={})

Args:
  - entry_a, entry_b (Any): Pydantic objects to compare
  - threshold (float): Minimum score for nested object equivalence (0.0-100.0)
  - property_scores (dict): Mutable dict populated with per-field scores and aggregates
  - max_depth (int): Recursion depth for comparing referenced objects (default 1)
  - weight_dict (dict): Custom field weights; weight=0 excludes field (default equal)

Returns:
  float: Similarity score 0.0-100.0

Field comparison strategies:
  - Strings: Semantic embedding similarity (via txtai)
  - ID references at depth > 0: Recursive object_similarity
  - Booleans, ints, floats: Exact match (100.0 or 0.0)
  - Lists: Jaccard similarity (intersection/union)
  - None values: Both None=100%, one None=0%

property_scores dict structure:
  {
    "field_name": {
      "score": 95.5,              # Score for this field (0.0-100.0)
      "weight": 1.0,              # Weight applied to this field
      "contributing_score": 95.5, # score * weight
    },
    "_sum_of_weights": 42.0,      # Total weight across scored fields
    "_final_score": 87.3,         # Weighted average across all fields
  }

---
=== object_equivalence ===
Determine if two objects are semantically equivalent based on threshold.

Signature:
  object_equivalence(entry_a, entry_b, threshold, property_scores, max_depth=1, weight_dict={})

Args:
  Same as object_similarity

Returns:
  bool: True if similarity >= threshold, False otherwise
""")

## Similarity API Reference

In [ ]:
# Find potentially duplicate risks (similarity >= 85%) across taxonomies
all_risks = ox.get_all("risks")
duplicate_pairs = []

# Compare the IBM AI Atlas risk "atlas-harmful-output" against all others
reference_risk = ox.get_by_id(None, "atlas-harmful-output")
threshold = 85.0

print(f"Scanning {len(all_risks)} risks for possible duplicates of {reference_risk.id}...")
print(f"Threshold: {threshold}%")
print()

for other_risk in all_risks:
    if other_risk.id == reference_risk.id:
        continue
    
    scores = {}
    similarity = ox.object_similarity(
        reference_risk, 
        other_risk, 
        threshold=threshold,
        property_scores=scores
    )
    
    if similarity >= threshold:
        duplicate_pairs.append((other_risk.id, similarity))

if duplicate_pairs:
    print(f"Found {len(duplicate_pairs)} potential duplicates:\n")
    for risk_id, sim in sorted(duplicate_pairs, key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {risk_id:30} | Similarity: {sim:.2f}%")
else:
    print(f"No duplicates found at {threshold}% threshold")

### Practical Use Cases

**Finding duplicate or near-duplicate risks across taxonomies:**

In [ ]:
# Compare at different depths
risks = ox.get_all("risks")[:5]  # Use a sample of risks
r1, r2 = risks[0], risks[1]

# Depth 0: only direct field comparison (no recursion into referenced objects)
scores_d0 = {}
sim_d0 = ox.object_similarity(
    r1, r2,
    threshold=50.0,
    property_scores=scores_d0,
    max_depth=0
)

# Depth 1: include comparison of objects referenced by ID fields
scores_d1 = {}
sim_d1 = ox.object_similarity(
    r1, r2,
    threshold=50.0,
    property_scores=scores_d1,
    max_depth=1
)

print(f"Similarity comparison at different recursion depths:")
print(f"  {r1.id} vs {r2.id}")
print()
print(f"  max_depth=0 (direct fields only):     {sim_d0:.2f}%")
print(f"  max_depth=1 (include references):     {sim_d1:.2f}%")
print()
print("Note: Depth=1 can find higher similarity if objects share referenced entities")
print("      (e.g., risks from the same taxonomy, group, or with shared mappings)")

### Recursion Depth: Compare Referenced Objects

With `max_depth > 0`, similarity includes comparison of objects referenced by ID fields:

In [ ]:
# Default: all fields equally weighted
scores_default = {}
sim_default = ox.object_similarity(
    r1, r2, 
    threshold=50.0, 
    property_scores=scores_default
)

print(f"Similarity (default weights): {sim_default:.2f}%")
print()

# Custom: prioritize name and description, ignore id and url
weight_dict = {
    "name": 10.0,          # High weight — very important
    "description": 5.0,    # Medium weight
    "id": 0,               # Exclude id from comparison
    "url": 0,              # Exclude url
}

scores_weighted = {}
sim_weighted = ox.object_similarity(
    r1, r2,
    threshold=50.0,
    property_scores=scores_weighted,
    weight_dict=weight_dict
)

print(f"Similarity (custom weights):")
print(f"  - name weight = 10.0 (high priority)")
print(f"  - description weight = 5.0 (medium)")
print(f"  - id, url excluded (weight = 0)")
print(f"  Result: {sim_weighted:.2f}%")
print()

# Show fields that were scored
weighted_fields = [f for f in scores_weighted if not f.startswith('_')]
print(f"Fields scored with custom weights: {len(weighted_fields)}")
print(f"'id' in scores: {'id' in scores_weighted}")
print(f"'name' in scores: {'name' in scores_weighted}")

### Custom Weights: Prioritize Important Fields

Control which fields matter most by providing custom weights:

In [ ]:
# Test equivalence at different thresholds
risks = ox.get_all("risks")
risk_a, risk_b = risks[0], risks[1]

scores_eq = {}
sim = ox.object_similarity(risk_a, risk_b, threshold=50.0, property_scores=scores_eq)

print(f"Similarity between {risk_a.id} and {risk_b.id}: {sim:.2f}%")
print()

# Test equivalence at different thresholds
thresholds = [50.0, 75.0, 90.0, 95.0]
print("Equivalence at different thresholds:")

for threshold in thresholds:
    equiv = ox.object_equivalence(risk_a, risk_b, threshold=threshold, property_scores={})
    status = "✓ EQUIVALENT" if equiv else "✗ NOT equivalent"
    print(f"  Threshold {threshold:5.1f}%: {status}")

### Equivalence: Threshold-Based Classification

Use `object_equivalence()` to classify whether two objects are "equivalent" based on a similarity threshold:

In [ ]:
# Inspect individual field scores from the comparison
scores_detail = {}
ox.object_similarity(r1, r2, threshold=0.5, property_scores=scores_detail)

print("Field-by-field similarity scores:")
print("Field            | Score  | Weight | Contributing")
print("-" * 50)

field_scores = [(k, v) for k, v in scores_detail.items() if not k.startswith('_')]
for field_name, field_data in sorted(field_scores)[:10]:
    score = field_data["score"]
    weight = field_data["weight"]
    contrib = field_data["contributing_score"]
    print(f"{field_name:16} | {score:6.1f} | {weight:6.1f} | {contrib:6.1f}")

### Inspecting Per-Field Scores

The `property_scores` dict shows you how similarity was computed across all fields:

In [ ]:
# Compare an object to itself (should be 100% similar)
scores = {}
similarity_same = ox.object_similarity(r1, r1, threshold=0.5, property_scores=scores)

print(f"Similarity (risk1 vs risk1):")
print(f"  Score: {similarity_same:.2f}%")
print(f"  Final score: {scores['_final_score']:.2f}%")
print(f"  Sum of weights: {scores['_sum_of_weights']:.1f}")
print()

# Compare two different risks
scores2 = {}
similarity_diff = ox.object_similarity(r1, r2, threshold=0.5, property_scores=scores2)

print(f"Similarity (risk1 vs risk2):")
print(f"  Score: {similarity_diff:.2f}%")
print(f"  Fields scored: {len([f for f in scores2 if not f.startswith('_')])}")

### Basic Similarity Comparison

Compare two objects and get a similarity score (0.0–100.0):

In [ ]:
# Get two risks to compare
risks = ox.get_all("risks")
risk1, risk2 = risks[0], risks[1]

print(f"Comparing two risks:")
print(f"  Risk 1: {risk1.id} - {risk1.name}")
print(f"  Risk 2: {risk2.id} - {risk2.name}")

## Part 5: Semantic Similarity — Comparing Ontology Entries

PyoxigraphExplorer now includes powerful methods for measuring how similar two ontology entries are:

## Part 7: Star Graph & Subgraph Comparison

PyoxigraphExplorer provides methods to extract and compare star graphs (1-hop neighborhoods) of entities in the knowledge graph. This is useful for understanding how two risks or actions relate to overlapping sets of neighbors.

In [ ]:
# Build a star graph for the first risk
risk_id_a = risks[0].id
star_graph_a = ox.get_star_graph(risk_id_a)

print(f"Star Graph for {risk_id_a}:")
print(f"  Center: {star_graph_a.center.name}")
print(f"  Total neighbors: {len(star_graph_a.neighbor_ids)}")
print()
print(f"  Neighbors by relationship type:")
for field_name, neighbors in star_graph_a.neighbors.items():
    print(f"    {field_name:20} : {len(neighbors):3} neighbors")

In [ ]:
# Build a star graph for a second risk
risk_id_b = risks[1].id
star_graph_b = ox.get_star_graph(risk_id_b)

print(f"Star Graph for {risk_id_b}:")
print(f"  Center: {star_graph_b.center.name}")
print(f"  Total neighbors: {len(star_graph_b.neighbor_ids)}")
print()
print(f"  Neighbors by relationship type:")
for field_name, neighbors in star_graph_b.neighbors.items():
    print(f"    {field_name:20} : {len(neighbors):3} neighbors")

In [ ]:
# Compare the two star graphs
comparison = ox.compare_star_graphs(risk_id_a, risk_id_b)

print(f"Star Graph Comparison: {risk_id_a} vs {risk_id_b}")
print()
print(f"  Jaccard Similarity: {comparison.jaccard_similarity:.4f} ({comparison.jaccard_similarity*100:.2f}%)")
print()
print(f"  Shared neighbors: {len(comparison.shared_neighbor_ids)}")
if comparison.shared_neighbor_ids:
    print(f"    Examples: {', '.join(list(comparison.shared_neighbor_ids)[:3])}")
print()
print(f"  Unique to {risk_id_a}: {len(comparison.unique_to_a)}")
if comparison.unique_to_a:
    print(f"    Examples: {', '.join(list(comparison.unique_to_a)[:3])}")
print()
print(f"  Unique to {risk_id_b}: {len(comparison.unique_to_b)}")
if comparison.unique_to_b:
    print(f"    Examples: {', '.join(list(comparison.unique_to_b)[:3])}")

In [ ]:
# Per-field breakdown: how neighbors differ by relationship type
print("\nPer-field breakdown (shared/unique neighbors by relationship type):")
print("=" * 80)
for field_name, breakdown in comparison.per_field.items():
    shared_count = len(breakdown['shared'])
    unique_a_count = len(breakdown['unique_to_a'])
    unique_b_count = len(breakdown['unique_to_b'])
    if shared_count > 0 or unique_a_count > 0 or unique_b_count > 0:
        print(f"\n{field_name}:")
        print(f"  Shared:     {shared_count}")
        print(f"  Only in A:  {unique_a_count}")
        print(f"  Only in B:  {unique_b_count}")

## Part 8: Natural Language Queries

`natural_language_query()` lets you query the knowledge graph in plain English. It uses an LLM inference engine to translate your question into SPARQL, then executes the query against the store and returns results in the same format as `sparql_query()`.

**How it works:**
1. The method builds a prompt that includes the graph schema (classes, predicates, few-shot SPARQL examples)
2. The LLM generates a SPARQL SELECT query
3. The query is executed on the pyoxigraph store
4. Results are returned as `list[dict]` — same format as `sparql_query()`

**Requirements:** Any configured `InferenceEngine` (Ollama, RITS, WML, vLLM, HF). The examples below use Ollama with `granite3.3:8b` running locally.

In [ ]:
from ai_atlas_nexus.blocks.inference import OllamaInferenceEngine

# Configure an inference engine — swap for RITSInferenceEngine, WMLInferenceEngine, etc.
# Requires Ollama running locally: https://ollama.com
engine = OllamaInferenceEngine(
    model_name_or_path="granite3.3:8b",
    credentials={"api_url": "http://localhost:11434"},
)

print("Inference engine ready:", type(engine).__name__)

In [ ]:
# Basic example: find all risks
question = "Find all risks"
results = ox.natural_language_query(question, engine)

print(f"Question: {question!r}")
print(f"Results: {len(results)} rows")
print()

# Resolve IDs back to Pydantic objects
resolved = [r for r in [ox._uri_to_pydantic(solution["?s"]) for solution in results if "?s" in solution] if r is not None]
print(f"Resolved {len(resolved)} risk objects")
if resolved:
    print(f"First result: {resolved[0].id} — {resolved[0].name}")

In [ ]:
# More specific question with a filter
question = "Find all actions from the NIST AI RMF taxonomy"
results = ox.natural_language_query(question, engine)

print(f"Question: {question!r}")
print(f"Results: {len(results)} rows")
if results:
    ids = [r.id for r in [ox._uri_to_pydantic(solution["?s"]) for solution in results if "?s" in solution] if r is not None] 
    print(f"Sample IDs: {ids}")

In [ ]:
# Relationship query: actions and the risks they address
question = "Which actions address which risks? Show me action-risk pairs defined by the NIST AI RMF taxonomy "
results = ox.natural_language_query(question, engine)

print(f"Question: {question!r}")
print(f"Results: {len(results)} action-risk pairs")
print()

if results:
    print("Sample pairs:")
    for row in results[:5]:
        # Keys depend on the variable names the LLM chose
        keys = list(row.keys())
        vals = [row[k].split("/")[-1] for k in keys if row.get(k)]
        print(f"  {' -> '.join(vals)}")

### Notes on natural_language_query

- **Result format** is the same as `sparql_query()`: a `list[dict]` where keys are SPARQL variable names and values are URI strings or literal values. Use `ox.get_by_id(None, id)` to resolve URIs to Pydantic objects.
- **Graceful failure**: if the LLM generates invalid SPARQL, the method logs a warning and returns `[]` rather than raising an exception.
- **SPARQL knowledge is not required** — but understanding the result structure helps when resolving objects. The variable names in results (`"s"`, `"action"`, `"risk"`, etc.) are chosen by the LLM and may vary.
- **Engine agnostic**: swap `OllamaInferenceEngine` for any other supported engine (RITS, WML, vLLM, HF) without changing the call site.

In [ ]:
ox.sparql_query(ox._qb.get_neighbours_with_hops("MG-4.3-003"))

results = ox.sparql_query(ox._qb.get_neighbours_with_hops("MG-4.3-003"))
print(f"First 5  (raw SPARQL results):")
for r in results:
    uri = r['?s']
    # Extract ID from URI
    risk_id = uri.split('/')[-1].rstrip('>')
    print(f"  {risk_id}")